# Task 29: AccessAudit QLoRA Training with Colab Persistence


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

DRIVE_BASE = "/content/drive/MyDrive/AccessAudit_Task29"
os.makedirs(os.path.join(DRIVE_BASE, "outputs"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_BASE, "lora_adapter"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_BASE, "gguf"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_BASE, "metadata"), exist_ok=True)

print(f"Created persistent directories at {DRIVE_BASE}")


In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps peft accelerate bitsandbytes datasets trl


In [ ]:
import torch
import unsloth
import transformers
import peft
import trl

print("--- GPU INFO ---")
if not torch.cuda.is_available():
    raise Exception("FAIL: NO CUDA GPU AVAILABLE! Please enable GPU in Runtime settings.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

print("\n--- LIB VERSIONS ---")
print("PyTorch:", torch.__version__)
print("Unsloth:", unsloth.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
try:
    import bitsandbytes
    print("BitsAndBytes:", bitsandbytes.__version__)
except ImportError:
    print("BitsAndBytes: NOT INSTALLED")
try:
    import xformers
    print("xformers:", xformers.__version__)
except ImportError:
    print("xformers: NOT INSTALLED (Optional)")


In [ ]:
from google.colab import files
import os

train_path = "/content/train.jsonl"
val_path = "/content/validation.jsonl"

missing_files = []
if not os.path.exists(train_path): missing_files.append("train.jsonl")
if not os.path.exists(val_path): missing_files.append("validation.jsonl")

if missing_files:
    print(f"Missing datasets in /content: {missing_files}. Please upload them now.")
    uploaded = files.upload()
    for fn in uploaded.keys():
        os.rename(fn, f"/content/{fn}")
else:
    print("Datasets already present in /content")


In [ ]:
import hashlib

def hash_file(path):
    if not os.path.exists(path): return "NOT FOUND"
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

expected_train_hash = "c9b48c7b0c70df2b8ffe95a5fb615ac353ecb54cb9a654897904edec56493bf0"
expected_val_hash = "bd4b6420d43984d9cddbc6eb2d341347a328ce74d0da0ba4ee9de813c90d6c26"

t_hash = hash_file(train_path)
v_hash = hash_file(val_path)
print("train.jsonl hash:", t_hash)
print("validation.jsonl hash:", v_hash)

if t_hash != expected_train_hash:
    raise Exception(f"Train hash mismatch! Expected {expected_train_hash}, got {t_hash}")
if v_hash != expected_val_hash:
    raise Exception(f"Validation hash mismatch! Expected {expected_val_hash}, got {v_hash}")

print("Data integrity confirmed.")


In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/qwen2.5-coder-7b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("/content/train.jsonl")
val_data = load_jsonl("/content/validation.jsonl")

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"}
)

def format_chat(examples):
    formatted = []
    for msgs in examples['messages']:
        formatted.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": formatted}

train_ds = Dataset.from_list(train_data).map(format_chat, batched=True)
val_ds = Dataset.from_list(val_data).map(format_chat, batched=True)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",  # Kept in /content for fast I/O
        eval_strategy = "epoch",
        save_strategy = "epoch",
    ),
)


In [ ]:
import time
start_time = time.time()
trainer_stats = trainer.train()
end_time = time.time()

duration = end_time - start_time
print(f"\nTraining Duration: {duration/60:.2f} minutes")


In [ ]:
lora_dir = os.path.join(DRIVE_BASE, "lora_adapter")
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f"LoRA adapter saved to {lora_dir}")


In [ ]:
print("Exporting to GGUF (Q4_K_M) locally to /content...")
try:
    model.save_pretrained_gguf("/content/qwen2.5-coder-7b-accessaudit", tokenizer, quantization_method = "q4_k_m")
    print("GGUF export successful in /content!")
except Exception as e:
    raise Exception(f"GGUF export failed: {e}")


In [ ]:
import shutil
import glob

# Copy GGUF
gguf_files = glob.glob("/content/**/*.gguf", recursive=True)
if not gguf_files:
    raise Exception("ERROR: No GGUF files found in /content after export!")

source_gguf = gguf_files[0]
dest_gguf = os.path.join(DRIVE_BASE, "gguf", os.path.basename(source_gguf))

print(f"Copying {source_gguf} to {dest_gguf} ...")
shutil.copy2(source_gguf, dest_gguf)
print("GGUF copy completed.")

# Copy outputs/checkpoints
source_outputs = "/content/outputs"
dest_outputs = os.path.join(DRIVE_BASE, "outputs")
if os.path.exists(source_outputs):
    print("Copying checkpoints to Google Drive...")
    shutil.copytree(source_outputs, dest_outputs, dirs_exist_ok=True)
    print("Checkpoints copy completed.")
else:
    print("No outputs/ folder found in /content to copy.")


In [ ]:
import json
import torch
import transformers
import peft
import unsloth
import trl

metadata = {
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if torch.cuda.is_available() else 0,
    "dataset_train_hash": expected_train_hash,
    "dataset_val_hash": expected_val_hash,
    "training_duration_seconds": duration,
    "train_loss": trainer_stats.metrics.get("train_loss"),
    "epochs": 2,
    "learning_rate": 2e-4,
    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "unsloth_version": unsloth.__version__,
    "peft_version": peft.__version__,
    "trl_version": trl.__version__
}

meta_path = os.path.join(DRIVE_BASE, "metadata", "task29_training_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

summary = f'''# Task 29 Cloud Training Summary
- **GPU**: {metadata['gpu']} ({metadata['vram_gb']} GB)
- **Duration**: {duration/60:.2f} minutes
- **Train Loss**: {metadata['train_loss']}
'''

sum_path = os.path.join(DRIVE_BASE, "metadata", "task29_training_summary.md")
with open(sum_path, "w") as f:
    f.write(summary)

print(f"Metadata saved to {DRIVE_BASE}/metadata")


In [ ]:
import glob
import os
import hashlib

print("\n-----------------------------------------------------")
print("VERIFYING ARTIFACTS ON GOOGLE DRIVE")
print("-----------------------------------------------------\n")

def get_size_str(b):
    for unit in ['B','KB','MB','GB','TB']:
        if b < 1024.0:
            return f"{b:.2f} {unit}"
        b /= 1024.0
    return f"{b:.2f} TB"

required_paths = [
    os.path.join(DRIVE_BASE, "lora_adapter", "adapter_model.safetensors"),
    os.path.join(DRIVE_BASE, "lora_adapter", "adapter_config.json"),
    os.path.join(DRIVE_BASE, "metadata", "task29_training_metadata.json"),
    os.path.join(DRIVE_BASE, "metadata", "task29_training_summary.md")
]

# Ensure at least one GGUF is in the gguf folder
drive_ggufs = glob.glob(os.path.join(DRIVE_BASE, "gguf", "*.gguf"))
if not drive_ggufs:
    raise Exception("ERROR: GGUF file is missing from Google Drive!")

required_paths.extend(drive_ggufs)

all_ok = True
print("ARTIFACT MANIFEST:")
for p in required_paths:
    if os.path.exists(p):
        size_bytes = os.path.getsize(p)
        print(f"✅ {os.path.basename(p)} - {get_size_str(size_bytes)}")
    else:
        print(f"❌ {os.path.basename(p)} - MISSING!")
        all_ok = False

if not all_ok:
    raise Exception("ERROR: Some required artifacts are missing from Google Drive.")

print("\n-----------------------------------------------------")
print("TASK 29 COMPLETE")
print("-----------------------------------------------------")
